# Buffett Letters: executable search notebook

Run this notebook from the extracted companion folder. Use Python 3.12 with the dependencies in `requirements.txt`. The original notebook is preserved in `dist/downloads/`.

The first preparation downloads the official letters and model. Later runs reuse the prepared Chroma index. The notebook is executable in Jupyter or VS Code; a Page itself has no Python kernel.

In [1]:
# If needed, uncomment this line, run it, then restart the kernel:
# %pip install -r requirements.txt
from pathlib import Path
import json
from prepare_corpus import ROOT, download_corpus, build_index
from serve import SearchService
print('Working folder:', ROOT)


Working folder: /Users/roshanxavier/Desktop/jenkins0screenshot/buffett_letters_app


## Prepare or reuse the index

In [2]:
if not (ROOT / 'work/index_info.json').exists():
    corpus = download_corpus()
    build_index(corpus)
info = json.loads((ROOT / 'work/index_info.json').read_text())
print(f"{len(info['documents'])} letters; {info['passages']:,} passages; {len(info['missing'])} missing years")
print('Reporting years:', min(d['year'] for d in info['documents']), '–', max(d['year'] for d in info['documents']))
print('Model revision:', info['revision'])
assert not info['missing'], 'Resolve missing years before treating the collection as complete'


49 letters; 4,539 passages; 0 missing years
Reporting years: 1977 – 2025
Model revision: 751bff37182d3f1213fa05d7196b954e230abad9


## Search with year, author and exact-text filters

In [3]:
service = SearchService()
corpus = json.loads((ROOT / 'dist/corpus.json').read_text())
by_id = {c['id']: c for c in corpus['chunks']}

def ask_buffett(query, start_year=1977, end_year=2024, author='Warren Buffett', contains='', n_results=6):
    response = service.search({'query':query, 'from':start_year, 'to':end_year,
                               'author':author, 'contains':contains})
    rows = []
    for hit in response['results'][:n_results]:
        passage = by_id[hit['id']]
        url = passage['url'] + (f"#page={passage['page']}" if passage['page'] else '')
        rows.append({**passage, 'relevance':round(hit['score'],4), 'source':url})
    return rows

def show_results(rows):
    for row in rows:
        print(f"\n{row['year']} · {row['author']} · relevance {row['relevance']}")
        print(row['text'])
        print(row['source'])

results = ask_buffett('Why is insurance float valuable?', n_results=3)
show_results(results)



1997 · Warren Buffett · relevance 0.7558
is money we hold but don't own. In an insurance operation, float arises because premiums are received before losses are paid, an interval that sometimes extends over many years. During that time, the insurer invests the money. Typically, this pleasant activity carries with it a downside: The premiums that an insurer takes in usually do not cover the losses and expenses it eventually must pay. That leaves it running an "underwriting loss," which is the cost of float. An insurance business has value if its cost of float over time is less than the cost the company would otherwise incur to obtain funds. But the business is a lemon if its cost of float is higher than market rates for money. A caution is appropriate here: Because loss costs must be estimated, insurers have enormous latitude in figuring their underwriting results, and that makes it very difficult for investors to calculate a company's true cost of float. Estimating errors, usually
htt

## Compare eras

In [4]:
for start, end in [(1977,1995),(2010,2024)]:
    print(f"\n=== Share repurchases: {start}–{end} ===")
    show_results(ask_buffett('When should a company repurchase its shares?', start, end, n_results=2))



=== Share repurchases: 1977–1995 ===

1984 · Warren Buffett · relevance 0.6188
and not always understood. The obvious point involves basic arithmetic: major repurchases at prices well below per-share intrinsic business value immediately increase, in a highly significant way, that value. When companies purchase their own stock, they often find it easy to get $2 of present value for $1. Corporate acquisition programs almost never do as well and, in a discouragingly large number of cases, fail to get anything close to $1 of value for each $1 expended. The other benefit of repurchases is less subject to precise measurement but can be fully as important over time. By making repurchases when a company’s market value is well below its business value, management clearly demonstrates that it is given to actions that enhance the wealth of shareholders, rather than to actions that expand management’s domain but that do nothing for (or even harm) shareholders. Seeing this, shareholders and potent

## Other authors and exact-text matching

The 2025 letter is by Greg Abel. Munger’s essay is in the 2014 anniversary PDF.

In [5]:
show_results(ask_buffett('How should capital be allocated?', 2025, 2025, author='Greg Abel', n_results=2))
show_results(ask_buffett('How was Berkshire built?', 2014, 2014, author='Charlie Munger', n_results=2))
show_results(ask_buffett('Insurance underwriting economics', contains='GEICO', n_results=2))



2025 · Greg Abel · relevance 0.4361
is to ensure our liquidity levels and capital deployment remain intentional and deliberate. We will always aim for ownership of productive businesses over U.S. Treasuries. Capital Discipline We deploy our shareholders’ capital to opportunities that generate rewards commensurate with their risk. When we expand existing operations, acquire new operating businesses, invest in equity securities, and repurchase Berkshire stock, we evaluate each opportunity based on its potential to grow Berkshire’s intrinsic value per share over a time horizon measured in perpetuity. Berkshire’s capital allocation principles and strategy guide us in identifying opportunities: • Invest in businesses that we thoroughly understand, with durable advantages and long-term economic prospects; • Partner with high integrity leaders who understand their customers and act like owners; 5
https://www.berkshirehathaway.com/letters/2025ltr.pdf#page=5

2025 · Greg Abel · relevance 0.388

## Export your results

In [6]:
import csv
output = ROOT / 'search_results.csv'
with output.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=['year','author','page','relevance','source','text'], extrasaction='ignore')
    writer.writeheader()
    writer.writerows(results)
print('Saved:', output)


Saved: /Users/roshanxavier/Desktop/jenkins0screenshot/buffett_letters_app/search_results.csv


## Interpretation and limitations

Relevance is cosine similarity, not a confidence probability or fact check. Results are passages, not generated answers. Reporting years usually precede publication dates. The model can truncate long passages at 256 tokens. Examine original context and authorship before quoting.

For the local web interface, run `python serve.py` in a terminal and open the displayed local address. The browser-hosted version uses the same exported corpus and model; the local server uses Python and ChromaDB.